# Tutorial: Correlation and Synergy Analysis

In this tutorial, we analyze a cohort of phenopackets corresponding to the gene **FBN1**. The data is retrieved from the [phenopacket store](https://github.com/monarch-initiative/phenopacket-store).

Using this dataset, we demonstrate a complete workflow with **ppkt2synergy**, including dataset construction, correlation analysis, and synergy analysis of phenotypic features.

We recommend running this tutorial in a **Jupyter notebook** for interactive analysis and visualization, although it can also be executed as a standard Python script.

---

## 1. Load phenopackets

We start by loading phenopackets for the **FBN1** cohort from the [phenopacket store](https://github.com/monarch-initiative/phenopacket-store).


In [5]:
from ppkt2synergy import load_phenopackets_by_cohort

phenopackets = load_phenopackets_by_cohort("FBN1")

print(f"Loaded {len(phenopackets)} phenopackets")

Loaded 143 phenopackets


You can specify any cohort or cohorts of interest available in the phenopacket store (e.g., "FBN1" in this example).

---

## 2. Build the dataset

Next, we convert phenopackets into a structured dataset suitable for statistical analysis.

In [6]:
from ppkt2synergy import PhenotypeDatasetBuilder
from gpsea.model import VariantEffect

dataset = PhenotypeDatasetBuilder(phenopackets).build(
    mane_tx_id="NM_000138.5",           # Transcript of interest (MANE Select)
    variant_effect_type=VariantEffect.MISSENSE_VARIANT, # Variant effect type
    missing_threshold=0.6              # Threshold for missing data
)

Individuals Processed: 100%|██████████| 143/143 [01:22<00:00,  1.73 individuals/s]


This step transforms raw phenotypic annotations into a structured dataset for downstream analysis.The dataset consists of:

* **HPO feature matrix**
  A binary matrix where rows correspond to individuals and columns correspond to HPO terms.
  Values indicate whether a phenotype is observed, excluded, or unknown.

* **Target variables**
  Additional matrices, such as disease labels or variant conditions, used for synergy analysis.

* **Individual metadata**
  Optional information about individual (e.g., cohort, sex, or publication identifiers).

**Key Parameters:**

* `mane_tx_id`: Specifies the transcript of interest. Here, we use the MANE Select transcript **NM_000138.5** for the *FBN1* gene.
* `variant_effect_type`: Defines the variant class to be considered. In this case, we're focusing on individuals with **missense_variant** mutation.
* `missing_threshold` controls how much missing data is allowed for each HPO feature. For example, a threshold of **0.6** means that features with up to 60% missing values to be retained.

After this step, the `dataset` object is ready for correlation and synergy analysis.

---

## 3. Correlation analysis

Next, we compute pairwise correlations between the HPO features.

In [ ]:
from ppkt2synergy import HPOCorrelationAnalyzer, CorrelationType

analyzer = HPOCorrelationAnalyzer(
    dataset=dataset,
    min_individuals_for_correlation_test=30,  # Minimum number of individuals required
    min_cooccurrence_count=1          # Minimum co-occurrence of features
)

analyzer.compute_correlation_matrix(
    correlation_type=CorrelationType.SPEARMAN,    # Choose correlation type (e.g., Spearman)
    n_jobs=32,         # Number of parallel jobs
)

Calculating pairwise correlation: 100%|██████████| 242/242 [00:07<00:00, 31.89it/s]


,HPO_A,HPO_A_label,HPO_B,HPO_B_label,correlation,p_value,p_value_corrected,Count_00,Count_01,Count_10,Count_11,n_individuals,n_pmids,pmids
182,HP:0004322,Short stature,HP:0004942,Aortic aneurysm,-0.827131,1.078983e-19,1.996118e-17,7,34,33,0,74,5,10756346;11175294;12203992;20375004;21683322
17,HP:0000098,Tall stature,HP:0004942,Aortic aneurysm,0.796637,1.652467e-18,1.528532e-16,45,7,1,26,79,13,10756346;11175294;12203992;20375004;20979188;2...
12,HP:0000098,Tall stature,HP:0002616,Aortic root aneurysm,0.646903,9.895588e-15,6.102280e-13,70,7,10,26,113,18,10756346;11175294;12203992;20375004;20979188;2...
16,HP:0000098,Tall stature,HP:0004322,Short stature,-0.685681,3.243534e-13,1.500135e-11,16,33,37,0,86,5,10756346;11175294;12203992;20375004;21683322
168,HP:0002616,Aortic root aneurysm,HP:0004322,Short stature,-0.634615,6.953208e-11,2.572687e-09,19,33,33,0,85,5,10756346;11175294;12203992;20375004;21683322
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
25,HP:0000218,High palate,HP:0001083,Ectopia lentis,0.013347,9.029263e-01,9.228805e-01,11,47,5,23,86,15,10756346;11175294;12203992;12446365;20979188;2...
112,HP:0001166,Arachnodactyly,HP:0001653,Mitral regurgitation,-0.015854,9.247326e-01,9.399755e-01,6,1,27,4,38,3,11175294;12203992;37845262
98,HP:0001083,Ectopia lentis,HP:0001382,Joint hypermobility,0.007778,9.433357e-01,9.504628e-01,11,6,44,25,86,13,10756346;11175294;12203992;12446365;20979188;2...
75,HP:0000767,Pectus excavatum,HP:0004942,Aortic aneurysm,0.009950,9.453252e-01,9.504628e-01,11,31,2,6,50,11,10756346;11175294;12203992;20375004;20979188;2...


This step calculates the pairwise correlations between all HPO terms across individuals in the cohort. The result is a matrix of correlation values, along with statistical information like p-values and adjusted p-values.

* **positive correlation** → the two features tend to appear together
* **negative correlation** → the features tend to occur in mutually exclusive individuals
* **values near zero** → little or no association

**Key Parameters:**
* `min_individuals_for_correlation_test`: Ensures that correlations are only computed when there are enough individuals.
* `min_cooccurrence_count`: Filters out feature pairs that rarely co-occur.
* `n_jobs`: Controls parallelization (higher values are better for multi-core systems).

The correlation results will help identify phenotype–phenotype relationships, such as co-occurring phenotypes or mutually exclusive features.

---

## 4. Visualize correlation results

We can visualize the correlation results as a heatmap.

In [8]:
fig = analyzer.plot_correlation_heatmap_with_significance(
    stats_name='spearman',
    abs_threshold=0.6,
    adj_pval_threshold=0.1,
    title_name="Cohort FBN1",
)
fig.show()

This visualization highlights the strongest and most statistically significant associations between phenotypic features.

**Key Parameters:**

* `abs_threshold`: Filters out weak correlations (only keeping stronger ones).
* `adj_pval_threshold`: Filters results based on statistical significance after multiple testing correction.

By adjusting these thresholds, you can control the sparsity of the heatmap and focus on the most relevant features.

---

## 5. Inspect available targets

Before running synergy analysis, it's helpful to inspect the available target variables in the dataset.

In [9]:
dataset.describe_available_targets()

{
  "built_targets": {
    "disease": {
      "available_target_names": [
        "Acromicric dysplasia",
        "Ectopia lentis, familial",
        "Geleophysic dysplasia 2",
        "Marfan lipodystrophy syndrome",
        "Marfan syndrome",
        "Stiff skin syndrome"
      ]
    },
    "variant_condition": {
      "available_target_names": [
        "missense_variant"
      ]
    }
  },
  "metadata_targets": {
    "sex": {
      "available_positive_classes": [
        "female",
        "male"
      ]
    },
    "cohort": {
      "available_positive_classes": [
        "FBN1"
      ]
    }
  }
}


This summary shows which targets (such as disease labels or variant conditions) can be used in subsequent analysis.

---

## 6. Synergy analysis

While correlation identifies pairwise relationships, synergy analysis looks for higher-order interactions between features.

We begin by initializing the synergy analyzer:

In [11]:
from ppkt2synergy import SynergyAnalyzer

synergy_analyzer = SynergyAnalyzer(
    dataset=dataset,
    n_permutations=1000,               # Number of permutation tests
    min_individuals_for_synergy_calculation=40,  # Minimum individuals required for synergy analysis
    random_state=40              # Ensure reproducibility
)

**Key Parameters:**

* `n_permutations`: Controls the number of permutations used for statistical significance.
* `min_individuals_for_synergy_calculation`: Ensures enough individuals are available for meaningful synergy analysis.
* `random_state`: Ensures reproducibility.


Next, we compute the synergy matrix for the selected target:

In [12]:
synergy_analyzer.compute_synergy_matrix(
    target_type="variant_condition", # Type of target (e.g., variant conditions)
    target_name="missense_variant",  # Specific target name (e.g., missense variant)
    n_jobs=32,                       # Number of parallel jobs
)

Calculating pairwise synergy: 100%|██████████| 219/219 [00:32<00:00,  6.84it/s]


,HPO_A,HPO_A_label,HPO_B,HPO_B_label,synergy,p_value,p_value_corrected,Count_00_y0,Count_01_y0,Count_10_y0,Count_11_y0,N_y0,Count_00_y1,Count_01_y1,Count_10_y1,Count_11_y1,N_y1,n_individuals,n_pmids,pmids
3,HP:0000098,Tall stature,HP:0000767,Pectus excavatum,0.081608,0.000999,0.024975,7,3,9,0,19,36,1,21,8,66,85,16,10756346;11175294;12203992;20375004;20979188;2...
27,HP:0000218,High palate,HP:0001187,Hyperextensibility of the finger joints,-0.061977,0.000999,0.024975,6,0,3,3,12,39,0,6,0,45,57,13,10756346;11175294;12203992;12446365;20979188;2...
128,HP:0001382,Joint hypermobility,HP:0004942,Aortic aneurysm,0.117675,0.000999,0.024975,1,7,2,2,12,31,13,0,11,55,67,7,10756346;11175294;12203992;20979188;21594992;2...
8,HP:0000098,Tall stature,HP:0001187,Hyperextensibility of the finger joints,0.041175,0.000999,0.024975,2,3,6,0,11,58,0,12,0,70,81,12,10756346;11175294;12203992;20979188;21594992;2...
112,HP:0001187,Hyperextensibility of the finger joints,HP:0001519,Disproportionate tall stature,0.028103,0.000999,0.024975,4,5,3,0,12,67,4,0,0,71,83,11,10756346;11175294;12203992;20979188;21594992;2...
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
44,HP:0000541,Retinal detachment,HP:0002650,Scoliosis,-0.000179,1.000000,1.000000,8,3,0,0,11,43,14,1,0,58,69,8,10756346;12203992;12446365;22219643;22539873;2...
57,HP:0000545,Myopia,HP:0004322,Short stature,0.000121,1.000000,1.000000,3,0,6,0,9,14,0,26,1,41,50,3,11175294;12203992;20375004
59,HP:0000545,Myopia,HP:0030053,Stiff skin,-0.015493,1.000000,1.000000,3,0,6,0,9,12,2,21,6,41,50,3,11175294;12203992;20375004
125,HP:0001382,Joint hypermobility,HP:0003502,Mild short stature,0.000300,1.000000,1.000000,9,0,3,0,12,44,2,19,0,65,77,4,10756346;11175294;12203992;21683322


Synergy values can be interpreted as follows:

* **positive synergy** → the two features jointly provide additional information about the target
* **near zero** → the features contribute largely independently
* **negative synergy** → the features are redundant with respect to the target

---

## 7. Visualize synergy results

We can visualize synergy results as a heatmap.

In [13]:
fig = synergy_analyzer.plot_synergy_heatmap(
    synergy_threshold=0.08,    # Threshold for synergy strength
    adj_pval_threshold=0.1,    # Adjusted p-value threshold
    target_name="missense_variant",
)
fig.show()

This visualization highlights feature pairs that show positive and statistically significant synergy with respect to the selected target.

**Key Parameters:**

* `synergy_threshold` filters out weak interactions and keeps only stronger synergistic effects
* `adj_pval_threshold` ensures that only statistically significant interactions are shown

---

## Summary

In this tutorial, we:

- Loaded phenopacket data  
- Constructed a structured dataset  
- Performed correlation analysis  
- Identified higher-order feature interactions using synergy  

For additional usage patterns and parameter options, see the **Usage** section.